In [1]:
import sys
import pandas as pd
import numpy as np
import pm4py

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PM4Py:", pm4py.__version__)
print("CareFlow environment setup successful!")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Python: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
Python executable: C:\Users\visha\.conda\envs\careflow\python.exe
Pandas: 3.0.6
NumPy: 2.4.6
PM4Py: 2.7.23.10
CareFlow environment setup successful!


In [7]:

from pathlib import Path
import random
import numpy as np
import pandas as pd
from datetime import timedelta

# ==========================================
# CAREFLOW: SYNTHETIC HOSPITAL EHR GENERATOR
# ==========================================

SEED = 42
N_CASES = 500

random.seed(SEED)
rng = np.random.default_rng(SEED)

ROOT = Path.cwd()

# If running from the notebooks folder, use its parent.
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp("2026-01-01 00:00:00")
END_DATE = pd.Timestamp("2026-09-30 23:59:59")

ACTIVITIES = [
    "Registration",
    "Triage",
    "Doctor Assessment",
    "Lab Order",
    "Lab Sample Collection",
    "Lab Result",
    "Imaging Order",
    "Imaging",
    "Treatment",
    "Reassessment",
    "Admission Decision",
    "Bed Assignment",
    "Discharge Preparation",
    "Discharge",
]

COMPLAINTS = [
    "Chest Pain", "Abdominal Pain", "Fever",
    "Breathing Difficulty", "Injury", "Headache",
    "Dizziness", "Vomiting", "Weakness",
    "Cough", "Back Pain", "Allergic Reaction",
]

DIAGNOSES = [
    "Chest Pain Under Evaluation",
    "Respiratory Infection",
    "Gastrointestinal Condition",
    "Minor Trauma",
    "Migraine",
    "Dehydration",
    "Musculoskeletal Pain",
    "Cardiac Condition Under Evaluation",
    "Metabolic Condition",
    "Other Unspecified Condition",
]

DEPARTMENTS = [
    "Emergency", "Laboratory", "Radiology",
    "General Medicine", "Cardiology",
    "Orthopedics", "Observation Unit",
]

RESOURCES = [
    ("DR001", "Doctor"),
    ("DR002", "Doctor"),
    ("DR003", "Doctor"),
    ("RN001", "Nurse"),
    ("RN002", "Nurse"),
    ("LAB001", "Lab Technician"),
    ("IMG001", "Radiology Technician"),
    ("ADM001", "Administrator"),
    ("BED001", "Bed Coordinator"),
]

def weighted_choice(values, weights):
    return random.choices(values, weights=weights, k=1)[0]

def make_case(case_number):
    case_id = f"CF-{case_number:07d}"
    patient_id = f"PAT-{case_number:07d}"

    age = int(np.clip(round(rng.normal(46, 22)), 0, 100))
    sex = weighted_choice(
        ["Female", "Male", "Other/Unknown"],
        [0.49, 0.49, 0.02]
    )

    acuity = weighted_choice(
        ["Low", "Moderate", "High", "Critical"],
        [0.30, 0.40, 0.23, 0.07]
    )

    arrival_mode = weighted_choice(
        ["Walk-in", "Ambulance", "Referral", "Transfer"],
        [0.58, 0.24, 0.12, 0.06]
    )

    complaint = random.choice(COMPLAINTS)
    diagnosis = random.choice(DIAGNOSES)
    comorbidities = random.sample(
        ["Diabetes", "Hypertension", "Asthma",
         "Heart Disease", "None", "Kidney Disease"],
        k=random.choices([1, 2, 3], weights=[0.60, 0.32, 0.08])[0]
    )
    if "None" in comorbidities and len(comorbidities) > 1:
        comorbidities.remove("None")

    arrival = START_DATE + pd.to_timedelta(
        int(rng.integers(0, int((END_DATE - START_DATE).total_seconds()))),
        unit="s"
    )

    case = {
        "case_id": case_id,
        "patient_id": patient_id,
        "age": age,
        "age_group": (
            "0-17" if age < 18 else
            "18-39" if age < 40 else
            "40-64" if age < 65 else "65+"
        ),
        "sex": sex,
        "region": random.choice(
            ["North", "South", "East", "West", "Central"]
        ),
        "arrival_mode": arrival_mode,
        "acuity": acuity,
        "chief_complaint": complaint,
        "diagnosis_group": diagnosis,
        "comorbidities": "|".join(comorbidities),
        "insurance_type": random.choice(
            ["Public", "Private", "Self-Pay", "Other"]
        ),
        "arrival_timestamp": arrival,
        "hospital_site": random.choice(
            ["Site-A", "Site-B", "Site-C"]
        ),
        "referral_source": random.choice(
            ["Self", "Primary Care", "Other Hospital", "Emergency Services"]
        ),
    }

    # The sequence intentionally includes realistic process variation.
    activities = [
        ("Registration", "Emergency"),
        ("Triage", "Emergency"),
        ("Doctor Assessment", "Emergency"),
    ]

    if random.random() < 0.72:
        activities += [
            ("Lab Order", "Emergency"),
            ("Lab Sample Collection", "Laboratory"),
            ("Lab Result", "Laboratory"),
        ]

    if random.random() < 0.38:
        activities += [
            ("Imaging Order", "Emergency"),
            ("Imaging", "Radiology"),
        ]

    activities.append(("Treatment", "Emergency"))

    # Deliberate loop-backs to simulate operational inefficiency.
    loopback_type = "None"
    if random.random() < 0.18:
        activities += [
            ("Reassessment", "Emergency"),
            ("Triage", "Emergency"),
        ]
        loopback_type = "Reassessment to Triage"

    if random.random() < 0.12:
        activities += [
            ("Doctor Assessment", "Emergency"),
        ]
        loopback_type = (
            loopback_type + "|Repeat Doctor Assessment"
            if loopback_type != "None"
            else "Repeat Doctor Assessment"
        )

    if acuity in ["High", "Critical"] and random.random() < 0.48:
        activities += [
            ("Admission Decision", "Emergency"),
            ("Bed Assignment", "Observation Unit"),
        ]
        disposition = "Admitted"
    else:
        disposition = weighted_choice(
            ["Discharged Home", "Transferred", "Left Before Completion"],
            [0.88, 0.08, 0.04]
        )

    activities.append(("Discharge Preparation", "Emergency"))
    activities.append(("Discharge", "Emergency"))

    # One event row per activity.
    event_rows = []
    current_time = arrival
    previous_activity = None

    for seq, (activity, department) in enumerate(activities, start=1):
        if seq > 1:
            # Waiting times intentionally have a long-tailed distribution.
            wait_minutes = int(np.clip(
                rng.lognormal(mean=2.5, sigma=0.9), 1, 720
            ))
            current_time += pd.Timedelta(minutes=wait_minutes)
        else:
            wait_minutes = 0

        duration_minutes = int(np.clip(
            rng.lognormal(mean=2.2, sigma=0.65), 1, 240
        ))

        resource_id, resource_role = random.choice(RESOURCES)

        event_rows.append({
            **case,
            "event_id": f"{case_id}-EV-{seq:03d}",
            "event_sequence": seq,
            "activity": activity,
            "timestamp": current_time,
            "previous_activity": previous_activity,
            "department": department,
            "resource_id": resource_id,
            "resource_role": resource_role,
            "location": random.choice(
                ["Zone-1", "Zone-2", "Zone-3", "Zone-4"]
            ),
            "wait_minutes": wait_minutes,
            "duration_minutes": duration_minutes,
            "loopback_type": loopback_type,
            "is_loopback": (
                activity == "Triage" and seq > 1
                or activity == "Doctor Assessment"
                and previous_activity == "Treatment"
            ),
            "event_status": weighted_choice(
                ["Completed", "Delayed", "Reworked"],
                [0.88, 0.09, 0.03]
            ),
            "disposition": disposition,
            "data_source": "Synthetic EHR Simulator",
            "data_version": "week1-v1",
        })

        previous_activity = activity
        current_time += pd.Timedelta(minutes=duration_minutes)

    case["disposition"] = disposition
    case["event_count"] = len(event_rows)
    case["loopback_type"] = loopback_type
    case["end_timestamp"] = current_time
    case["length_of_stay_minutes"] = int(
        (current_time - arrival).total_seconds() / 60
    )

    return case, event_rows


# Generate all cases.
case_rows = []
all_events = []

for i in range(1, N_CASES + 1):
    case, events = make_case(i)
    case_rows.append(case)
    all_events.extend(events)

cases_df = pd.DataFrame(case_rows)
events_df = pd.DataFrame(all_events)

# Add controlled, documented data-quality defects.
# These are deliberate and will be cleaned in a later step.
if len(events_df) > 100:
    missing_idx = rng.choice(events_df.index, size=50, replace=False)
    events_df.loc[missing_idx, "resource_id"] = None

    duplicate_rows = events_df.sample(
        n=25, random_state=SEED
    )
    events_df = pd.concat(
        [events_df, duplicate_rows], ignore_index=True
    )

# Save raw files. Do not overwrite these during cleaning.
events_path = RAW_DIR / "ehr_event_log_raw.csv"
cases_path = RAW_DIR / "ehr_cases_raw.csv"

events_df.to_csv(events_path, index=False)
cases_df.to_csv(cases_path, index=False)

print("CAREFLOW DATA GENERATION COMPLETE")
print("=" * 48)
print("Cases:", f"{len(cases_df):,}")
print("Event rows (including deliberate duplicates):",
      f"{len(events_df):,}")
print("Event columns:", len(events_df.columns))
print("Case columns:", len(cases_df.columns))
print("Unique cases in event log:",
      events_df["case_id"].nunique())
print("Event date range:",
      events_df["timestamp"].min(),
      "to", events_df["timestamp"].max())
print("Loopback cases:",
      cases_df["loopback_type"].ne("None").sum())
print("Missing resource IDs:",
      events_df["resource_id"].isna().sum())
print("Files saved:")
print(events_path.resolve())
print(cases_path.resolve())


CAREFLOW DATA GENERATION COMPLETE
Cases: 500
Event rows (including deliberate duplicates): 4,900
Event columns: 32
Case columns: 20
Unique cases in event log: 500
Event date range: 2026-01-01 00:00:20 to 2026-10-01 02:46:16
Loopback cases: 144
Missing resource IDs: 50
Files saved:
E:\careflow\data\raw\ehr_event_log_raw.csv
E:\careflow\data\raw\ehr_cases_raw.csv


In [8]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name.lower() == "notebooks":
    root = root.parent

events = pd.read_csv(root / "data/raw/ehr_event_log_raw.csv")
cases = pd.read_csv(root / "data/raw/ehr_cases_raw.csv")

print("EVENT LOG SHAPE:", events.shape)
print("CASE TABLE SHAPE:", cases.shape)

print("\nEVENT LOG COLUMNS:")
print(events.columns.tolist())

print("\nFIRST 10 EVENTS:")
display(events.head(10))

print("\nACTIVITY FREQUENCY:")
display(
    events["activity"]
    .value_counts()
    .rename_axis("activity")
    .reset_index(name="event_count")
)

print("\nEVENTS PER CASE:")
display(events.groupby("case_id").size().describe())

print("\nMISSING VALUES:")
display(
    events.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nDUPLICATE EVENT IDs:",
      events["event_id"].duplicated().sum())

EVENT LOG SHAPE: (4900, 32)
CASE TABLE SHAPE: (500, 20)

EVENT LOG COLUMNS:
['case_id', 'patient_id', 'age', 'age_group', 'sex', 'region', 'arrival_mode', 'acuity', 'chief_complaint', 'diagnosis_group', 'comorbidities', 'insurance_type', 'arrival_timestamp', 'hospital_site', 'referral_source', 'event_id', 'event_sequence', 'activity', 'timestamp', 'previous_activity', 'department', 'resource_id', 'resource_role', 'location', 'wait_minutes', 'duration_minutes', 'loopback_type', 'is_loopback', 'event_status', 'disposition', 'data_source', 'data_version']

FIRST 10 EVENTS:


,case_id,patient_id,age,age_group,sex,region,arrival_mode,acuity,chief_complaint,diagnosis_group,...,resource_role,location,wait_minutes,duration_minutes,loopback_type,is_loopback,event_status,disposition,data_source,data_version
0,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Bed Coordinator,Zone-4,0,14,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
1,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Nurse,Zone-1,28,2,NaN,True,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
2,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Doctor,Zone-4,3,9,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
3,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Doctor,Zone-2,9,8,NaN,False,Delayed,Discharged Home,Synthetic EHR Simulator,week1-v1
4,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Lab Technician,Zone-1,5,15,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
5,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Doctor,Zone-3,24,9,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
6,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Nurse,Zone-1,33,12,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
7,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Bed Coordinator,Zone-1,5,11,NaN,False,Reworked,Discharged Home,Synthetic EHR Simulator,week1-v1
8,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Radiology Technician,Zone-1,5,15,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1
9,CF-0000001,PAT-0000001,53,40-64,Male,North,Walk-in,Low,Breathing Difficulty,Gastrointestinal Condition,...,Lab Technician,Zone-2,11,8,NaN,False,Completed,Discharged Home,Synthetic EHR Simulator,week1-v1



ACTIVITY FREQUENCY:


,activity,event_count
0,Triage,584
1,Doctor Assessment,576
2,Treatment,503
3,Discharge,502
4,Registration,501
5,Discharge Preparation,500
6,Lab Sample Collection,377
7,Lab Order,376
8,Lab Result,374
9,Imaging,198



EVENTS PER CASE:


count    500.000000
mean       9.800000
std        1.975806
min        6.000000
25%        9.000000
50%       10.000000
75%       11.000000
max       16.000000
dtype: float64


MISSING VALUES:


loopback_type        3308
previous_activity     501
comorbidities         477
resource_id            50
sex                     0
case_id                 0
age                     0
patient_id              0
acuity                  0
chief_complaint         0
diagnosis_group         0
insurance_type          0
arrival_timestamp       0
region                  0
arrival_mode            0
dtype: int64


DUPLICATE EVENT IDs: 25


In [9]:

from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name.lower() == "notebooks":
    root = root.parent

events = pd.read_csv(root / "data/raw/ehr_event_log_raw.csv")
cases = pd.read_csv(root / "data/raw/ehr_cases_raw.csv")

events["timestamp"] = pd.to_datetime(events["timestamp"])
events["arrival_timestamp"] = pd.to_datetime(
    events["arrival_timestamp"]
)

print("1. UNIQUE IDENTIFIERS")
print("Unique case IDs:", events["case_id"].nunique())
print("Unique event IDs:", events["event_id"].nunique())
print("Duplicate event IDs:",
      events["event_id"].duplicated().sum())

print("\n2. FIRST EVENT CHECK")
ordered = events.sort_values(
    ["case_id", "timestamp", "event_sequence"]
)
first_events = ordered.groupby("case_id").first()
print("First event per case:")
print(first_events["activity"].value_counts())

print("\n3. TIMESTAMP CHECK")
print("Events before arrival:",
      (events["timestamp"] < events["arrival_timestamp"]).sum())
print("Maximum event timestamp:", events["timestamp"].max())

print("\n4. CASE RECONCILIATION")
print("Cases table:", len(cases))
print("Event-log unique cases:", events["case_id"].nunique())
print("Cases missing from event log:",
      len(set(cases["case_id"]) - set(events["case_id"])))

print("\n5. LOOP-BACK AUDIT")
print("Loop-back cases in case table:",
      cases["loopback_type"].ne("None").sum())
print("Rows flagged as loop-back:",
      events["is_loopback"].astype(str).str.lower().eq("true").sum())

print("\n6. QUALITY SUMMARY")
print("Event rows:", len(events))
print("Missing resource IDs:", events["resource_id"].isna().sum())
print("Missing previous activity:",
      events["previous_activity"].isna().sum())


1. UNIQUE IDENTIFIERS
Unique case IDs: 500
Unique event IDs: 4875
Duplicate event IDs: 25

2. FIRST EVENT CHECK
First event per case:
activity
Registration    500
Name: count, dtype: int64

3. TIMESTAMP CHECK
Events before arrival: 0
Maximum event timestamp: 2026-10-01 02:46:16

4. CASE RECONCILIATION
Cases table: 500
Event-log unique cases: 500
Cases missing from event log: 0

5. LOOP-BACK AUDIT
Loop-back cases in case table: 500
Rows flagged as loop-back: 649

6. QUALITY SUMMARY
Event rows: 4900
Missing resource IDs: 50
Missing previous activity: 501


In [10]:

import pandas as pd
from pathlib import Path

root = Path.cwd()
if root.name.lower() == "notebooks":
    root = root.parent

cases = pd.read_csv(root / "data/raw/ehr_cases_raw.csv")
events = pd.read_csv(root / "data/raw/ehr_event_log_raw.csv")

# Correct case-level count
loopback_labels = (
    cases["loopback_type"]
    .fillna("None")
    .astype(str)
    .str.strip()
)

case_loopback = loopback_labels.ne("None")

print("Total cases:", len(cases))
print("Cases with loop-backs:", case_loopback.sum())
print("Cases without loop-backs:", (~case_loopback).sum())

print("\nLoop-back types:")
print(loopback_labels.value_counts())

print("\nEvent-level loop-back flags:")
print(events["is_loopback"].value_counts(dropna=False))

print("\nDuplicate event IDs:",
      events["event_id"].duplicated().sum())


Total cases: 500
Cases with loop-backs: 144
Cases without loop-backs: 356

Loop-back types:
loopback_type
None                                               356
Reassessment to Triage                              70
Repeat Doctor Assessment                            65
Reassessment to Triage|Repeat Doctor Assessment      9
Name: count, dtype: int64

Event-level loop-back flags:
is_loopback
False    4251
True      649
Name: count, dtype: int64

Duplicate event IDs: 25


In [11]:

from pathlib import Path
import pandas as pd
import json

# ==========================================
# CAREFLOW: WEEK 1 DATA CLEANING PIPELINE
# ==========================================

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORT_DIR = ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load raw files
events = pd.read_csv(RAW_DIR / "ehr_event_log_raw.csv")
cases = pd.read_csv(RAW_DIR / "ehr_cases_raw.csv")

raw_event_rows = len(events)
raw_case_rows = len(cases)
raw_duplicate_ids = int(events["event_id"].duplicated().sum())
raw_missing_resources = int(events["resource_id"].isna().sum())

# 2. Validate required columns
required_event_columns = [
    "event_id", "case_id", "activity", "timestamp",
    "event_sequence", "resource_id"
]
required_case_columns = ["case_id", "arrival_timestamp"]

missing_event_columns = [
    c for c in required_event_columns if c not in events.columns
]
missing_case_columns = [
    c for c in required_case_columns if c not in cases.columns
]

if missing_event_columns or missing_case_columns:
    raise ValueError(
        f"Missing event columns: {missing_event_columns}; "
        f"missing case columns: {missing_case_columns}"
    )

# 3. Convert timestamp columns
events["timestamp"] = pd.to_datetime(
    events["timestamp"], errors="coerce"
)
events["arrival_timestamp"] = pd.to_datetime(
    events["arrival_timestamp"], errors="coerce"
)
cases["arrival_timestamp"] = pd.to_datetime(
    cases["arrival_timestamp"], errors="coerce"
)
cases["end_timestamp"] = pd.to_datetime(
    cases["end_timestamp"], errors="coerce"
)

invalid_event_timestamps = int(events["timestamp"].isna().sum())

# Rows without essential identifiers/timestamps cannot enter
# the cleaned event log. Preserve the original raw CSV.
events = events.dropna(
    subset=["event_id", "case_id", "activity", "timestamp"]
).copy()

# 4. Remove duplicate event IDs
events = events.drop_duplicates(
    subset=["event_id"], keep="first"
).copy()

# 5. Validate case references
valid_case_ids = set(cases["case_id"])
orphan_rows = int((~events["case_id"].isin(valid_case_ids)).sum())

events = events[
    events["case_id"].isin(valid_case_ids)
].copy()

# 6. Sort events into chronological case sequences
events = events.sort_values(
    ["case_id", "event_sequence", "timestamp"],
    kind="stable"
).reset_index(drop=True)

# Recompute previous_activity from the cleaned event sequence.
# The first event of each case should have no previous activity.
events["previous_activity"] = (
    events.groupby("case_id")["activity"].shift(1)
)

# 7. Handle intentional missing categorical values
events["resource_id"] = (
    events["resource_id"].fillna("Unknown")
)
events["loopback_type"] = (
    events["loopback_type"].fillna("None")
)
cases["loopback_type"] = (
    cases["loopback_type"].fillna("None")
)

# 8. Add a reproducible cleaned-data version
events["cleaning_version"] = "week1-v1"
cases["cleaning_version"] = "week1-v1"

# 9. Save processed datasets separately
event_output = PROCESSED_DIR / "ehr_event_log_clean.csv"
case_output = PROCESSED_DIR / "ehr_cases_clean.csv"

events.to_csv(event_output, index=False)
cases.to_csv(case_output, index=False)

# 10. Create a quality report
report = {
    "raw_event_rows": raw_event_rows,
    "clean_event_rows": len(events),
    "removed_duplicate_event_ids": raw_duplicate_ids,
    "invalid_event_timestamps": invalid_event_timestamps,
    "orphan_event_rows_removed": orphan_rows,
    "raw_missing_resource_ids": raw_missing_resources,
    "unknown_resource_ids_after_cleaning": int(
        events["resource_id"].eq("Unknown").sum()
    ),
    "clean_unique_event_ids": int(events["event_id"].nunique()),
    "clean_unique_case_ids": int(events["case_id"].nunique()),
    "case_table_rows": len(cases),
    "first_event_previous_activity_missing": int(
        events.groupby("case_id").head(1)["previous_activity"].isna().sum()
    ),
    "remaining_duplicate_event_ids": int(
        events["event_id"].duplicated().sum()
    ),
    "events_before_arrival": int(
        (events["timestamp"] < events["arrival_timestamp"]).sum()
    ),
}

report_path = REPORT_DIR / "week1_data_quality_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print("CAREFLOW CLEANING COMPLETE")
print("=" * 45)

for key, value in report.items():
    print(f"{key}: {value}")

print("\nSaved files:")
print(event_output.resolve())
print(case_output.resolve())
print(report_path.resolve())


CAREFLOW CLEANING COMPLETE
raw_event_rows: 4900
clean_event_rows: 4875
removed_duplicate_event_ids: 25
invalid_event_timestamps: 0
orphan_event_rows_removed: 0
raw_missing_resource_ids: 50
unknown_resource_ids_after_cleaning: 50
clean_unique_event_ids: 4875
clean_unique_case_ids: 500
case_table_rows: 500
first_event_previous_activity_missing: 500
remaining_duplicate_event_ids: 0
events_before_arrival: 0

Saved files:
E:\careflow\data\processed\ehr_event_log_clean.csv
E:\careflow\data\processed\ehr_cases_clean.csv
E:\careflow\reports\week1_data_quality_report.json


In [12]:

import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

events = pd.read_csv(
    ROOT / "data" / "processed" / "ehr_event_log_clean.csv"
)
cases = pd.read_csv(
    ROOT / "data" / "processed" / "ehr_cases_clean.csv"
)

assert events["event_id"].is_unique, "Duplicate event IDs remain!"
assert events["case_id"].isin(cases["case_id"]).all(), "Orphan cases found!"
assert events["timestamp"].notna().all(), "Invalid timestamps remain!"

first_events = (
    events.sort_values(["case_id", "event_sequence", "timestamp"])
    .groupby("case_id", sort=False)
    .head(1)
)

assert first_events["previous_activity"].isna().all(), (
    "Unexpected previous_activity on first event"
)

print("PASS: Event IDs are unique.")
print("PASS: All event cases exist in the case table.")
print("PASS: All event timestamps are valid.")
print("PASS: First events have no previous activity.")
print("Clean event rows:", len(events))
print("Clean case rows:", len(cases))


PASS: Event IDs are unique.
PASS: All event cases exist in the case table.
PASS: All event timestamps are valid.
PASS: First events have no previous activity.
Clean event rows: 4875
Clean case rows: 500
